# TRAVLR Revision Experiments Notebook

Notebook ini dibuat untuk menjalankan revisi eksperimen yang dibutuhkan paper:

1. **Calibration and threshold-sensitivity analysis** untuk klasifikasi high-demand.
2. **Temporal holdout and robustness analysis** untuk menguji generalisasi lintas waktu.
3. **Feature drift diagnostics** untuk memeriksa pergeseran distribusi fitur.
4. **Recommendation baseline comparison** untuk membandingkan proposed shortlisting dengan baseline sederhana.

Output utama akan disimpan ke folder `travlr_outputs/revision_experiments/` dan bisa langsung digunakan untuk mengganti placeholder table/figure pada paper.


In [ ]:
# ============================================================
# 0. Environment setup
# ============================================================
import os
import re
import json
import math
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path("travlr_outputs/revision_experiments")
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())


## 1. Download / locate dataset

Secara default notebook ini mencoba mengambil data dari KaggleHub. Jika dijalankan di lingkungan yang tidak memiliki akses internet/Kaggle, isi manual `DATASET_DIR` dengan folder dataset lokal.


In [ ]:
# ============================================================
# 1. Dataset download or manual path
# ============================================================

DATASET_DIR = None  # contoh manual: "/content/icoict-challenge-2026"

if DATASET_DIR is None:
    try:
        import kagglehub
        DATASET_DIR = kagglehub.dataset_download("noviananggis/icoict-challenge-2026")
        print("Dataset downloaded via kagglehub:", DATASET_DIR)
    except Exception as e:
        print("KaggleHub download failed. Set DATASET_DIR manually.")
        print("Error:", repr(e))
        DATASET_DIR = None

if DATASET_DIR is not None:
    DATASET_DIR = Path(DATASET_DIR)
    print("Dataset directory:", DATASET_DIR.resolve())
    print("Files:")
    for p in DATASET_DIR.rglob("*"):
        if p.is_file():
            print("-", p.relative_to(DATASET_DIR))
else:
    print("DATASET_DIR is None. Please set it manually and rerun this cell.")


In [ ]:
# ============================================================
# 2. Utility functions for reading and column handling
# ============================================================

def normalize_col(c: str) -> str:
    c = str(c).strip().lower()
    c = re.sub(r"[^a-z0-9]+", "_", c)
    c = re.sub(r"_+", "_", c).strip("_")
    return c

def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in [".csv", ".txt"]:
        return pd.read_csv(path)
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".json":
        return pd.read_json(path)
    raise ValueError(f"Unsupported file type: {path}")

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [normalize_col(c) for c in out.columns]
    return out

def find_col(df: pd.DataFrame, candidates: List[str], required: bool = False, default: Optional[str] = None) -> Optional[str]:
    cols = set(df.columns)
    normalized_candidates = [normalize_col(c) for c in candidates]
    for c in normalized_candidates:
        if c in cols:
            return c
    # fuzzy contains
    for cand in normalized_candidates:
        for col in df.columns:
            if cand in col or col in cand:
                return col
    if required:
        raise KeyError(f"None of the candidate columns found: {candidates}. Available: {list(df.columns)[:30]}")
    return default

def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)

def safe_numeric(df: pd.DataFrame, col: Optional[str], default=np.nan) -> pd.Series:
    if col and col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(default, index=df.index)

def infer_files(dataset_dir: Path) -> Dict[str, Path]:
    files = [p for p in dataset_dir.rglob("*") if p.is_file() and p.suffix.lower() in [".csv", ".xlsx", ".xls", ".parquet", ".json"]]
    scored = {"transactions": [], "metadata": [], "activity": []}
    for p in files:
        name = normalize_col(p.name)
        if any(k in name for k in ["transaction", "transactions", "booking"]):
            scored["transactions"].append(p)
        if any(k in name for k in ["accommodation", "accomodation", "hotel", "metadata", "property"]):
            scored["metadata"].append(p)
        if any(k in name for k in ["activity", "activities", "tour", "attraction"]):
            scored["activity"].append(p)
    # fallback by file size if ambiguous
    result = {}
    for key, candidates in scored.items():
        if candidates:
            result[key] = sorted(candidates, key=lambda x: x.stat().st_size, reverse=True)[0]
    return result


In [ ]:
# ============================================================
# 3. Load raw data
# ============================================================

if DATASET_DIR is None:
    raise RuntimeError("Please set DATASET_DIR first.")

files = infer_files(DATASET_DIR)
print("Inferred files:")
for k, v in files.items():
    print(k, "->", v)

# Manual override if inference is wrong:
TRANSACTIONS_PATH = files.get("transactions")
METADATA_PATH = files.get("metadata")
ACTIVITY_PATH = files.get("activity")

if TRANSACTIONS_PATH is None or METADATA_PATH is None or ACTIVITY_PATH is None:
    raise FileNotFoundError("Could not infer all required files. Please set TRANSACTIONS_PATH, METADATA_PATH, and ACTIVITY_PATH manually.")

tx_raw = standardize_columns(read_table(TRANSACTIONS_PATH))
meta_raw = standardize_columns(read_table(METADATA_PATH))
act_raw = standardize_columns(read_table(ACTIVITY_PATH))

print("Transactions:", tx_raw.shape)
print("Metadata:", meta_raw.shape)
print("Activity:", act_raw.shape)

display(tx_raw.head())
display(meta_raw.head())
display(act_raw.head())


In [ ]:
# ============================================================
# 4. Build analytical dataset
# ============================================================

tx = tx_raw.copy()
meta = meta_raw.copy()
act = act_raw.copy()

# ---- Transaction columns
tx_date_col = find_col(tx, ["payment_date", "date", "booking_date", "month"], required=True)
tx_pid_col = find_col(tx, ["property_id", "property id", "property", "hotel_id", "id"], required=True)
tx_product_col = find_col(tx, ["product", "hotel_name", "name", "property_name"], required=False)
tx_n_col = find_col(tx, ["number_transactions", "number of transactions", "transactions", "transaction_count", "count"], required=True)

tx["payment_month"] = pd.to_datetime(tx[tx_date_col].astype(str), errors="coerce", format="%b-%y")
if tx["payment_month"].isna().mean() > 0.5:
    tx["payment_month"] = pd.to_datetime(tx[tx_date_col].astype(str), errors="coerce")
tx["month"] = tx["payment_month"].dt.month.fillna(0).astype(int)
tx["number_transactions"] = pd.to_numeric(tx[tx_n_col], errors="coerce").fillna(0)
tx["property_id_clean"] = tx[tx_pid_col].astype(str).str.strip()
tx["provider_prefix"] = tx["property_id_clean"].str.split("-", n=1).str[0].str.lower()

# ---- Metadata columns
meta_pid_col = find_col(meta, ["property_id", "property id", "property", "hotel_id", "id", "listing_id"], required=False)
if meta_pid_col is not None:
    meta["property_id_clean"] = meta[meta_pid_col].astype(str).str.strip()
else:
    # If no explicit property id exists, create empty id; exact join will have low coverage.
    meta["property_id_clean"] = ""

meta_name_col = find_col(meta, ["product", "hotel_name", "name", "property_name", "title"], required=False)
meta_country_col = find_col(meta, ["country", "destination_country"], required=False)
meta_city_col = find_col(meta, ["city", "destination_city", "locality"], required=False)
meta_provider_col = find_col(meta, ["provider", "providers", "source"], required=False)

if meta_provider_col:
    meta["provider_prefix"] = meta[meta_provider_col].astype(str).str.lower()
else:
    meta["provider_prefix"] = meta["property_id_clean"].str.split("-", n=1).str[0].str.lower()

if meta_country_col:
    meta["country_clean"] = meta[meta_country_col].astype(str).str.lower().str.strip()
else:
    meta["country_clean"] = "unknown"

if meta_city_col:
    meta["city_clean"] = meta[meta_city_col].astype(str).str.lower().str.strip()
else:
    meta["city_clean"] = "unknown"

meta["dest_key"] = meta["country_clean"] + "||" + meta["city_clean"]

# useful metadata numeric columns
for new_col, candidates in {
    "final_price_aud": ["final_price_aud", "price_aud", "price", "final_price", "amount"],
    "guest_rating": ["guest_rating", "rating", "review_score"],
    "guest_rating_count": ["guest_rating_count", "rating_count", "review_count", "reviews"],
    "availability_score": ["availability_score", "availability", "available"],
    "popularity": ["popularity", "popularity_score"],
    "star_rating_score": ["star_rating_score", "star_rating", "stars"]
}.items():
    col = find_col(meta, candidates, required=False)
    if col:
        meta[new_col] = pd.to_numeric(meta[col].astype(str).str.extract(r"([0-9]+\\.?[0-9]*)")[0] if meta[col].dtype == object else meta[col], errors="coerce")
    else:
        meta[new_col] = np.nan

# deduplicate metadata by property id for exact join
meta_dedup = meta.sort_values("property_id_clean").drop_duplicates("property_id_clean", keep="first")

# exact transaction-metadata join
df = tx.merge(meta_dedup, on="property_id_clean", how="left", suffixes=("", "_meta"))
df["matched_metadata"] = df["country_clean"].notna() & (df["country_clean"].astype(str) != "nan")

# ---- Activity columns
act_country_col = find_col(act, ["country", "destination_country"], required=False)
act_city_col = find_col(act, ["city", "destination_city", "location_city"], required=False)
act_price_col = find_col(act, ["price_aud", "final_price_aud", "price", "amount"], required=False)
act_rating_col = find_col(act, ["normalized_rating", "rating", "review_score"], required=False)
act_avail_col = find_col(act, ["availability", "availability_score", "available"], required=False)
act_view_col = find_col(act, ["view_count", "views", "popularity", "number_views"], required=False)
act_cat_col = find_col(act, ["categories_as_string", "category", "categories"], required=False)
act_name_col = find_col(act, ["name", "activity_name", "title", "product"], required=False)

act["country_clean"] = act[act_country_col].astype(str).str.lower().str.strip() if act_country_col else "unknown"
act["city_clean"] = act[act_city_col].astype(str).str.lower().str.strip() if act_city_col else "unknown"
act["dest_key"] = act["country_clean"] + "||" + act["city_clean"]
act["activity_price"] = safe_numeric(act, act_price_col)
act["activity_rating"] = safe_numeric(act, act_rating_col)
act["activity_availability"] = safe_numeric(act, act_avail_col)
act["activity_views"] = safe_numeric(act, act_view_col).fillna(0)
act["activity_category"] = act[act_cat_col].astype(str) if act_cat_col else "unknown"
act["activity_name"] = act[act_name_col].astype(str) if act_name_col else act.index.astype(str)

# aggregate activities at destination level
activity_agg = (
    act.groupby("dest_key")
    .agg(
        activity_count=("activity_name", "count"),
        avg_activity_price=("activity_price", "mean"),
        avg_activity_rating=("activity_rating", "mean"),
        avg_activity_availability=("activity_availability", "mean"),
        total_activity_views=("activity_views", "sum"),
        dominant_activity_category=("activity_category", lambda x: x.mode().iloc[0] if len(x.mode()) else "unknown")
    )
    .reset_index()
)

df = df.merge(activity_agg, on="dest_key", how="left")
df["matched_activity"] = df["activity_count"].notna()

# derived features
df["log_price"] = np.log1p(pd.to_numeric(df["final_price_aud"], errors="coerce"))
df["rating_x_price"] = pd.to_numeric(df["guest_rating"], errors="coerce") * df["log_price"]
df["availability_x_popularity"] = pd.to_numeric(df["availability_score"], errors="coerce") * pd.to_numeric(df["popularity"], errors="coerce")
df["high_demand"] = (df["number_transactions"] > df["number_transactions"].median()).astype(int)

# Ensure categoricals exist
for c in ["country_clean", "city_clean", "provider_prefix", "dominant_activity_category"]:
    if c not in df.columns:
        df[c] = "unknown"
    df[c] = df[c].fillna("unknown").astype(str)

print("Analytical dataset:", df.shape)
print("Metadata coverage:", df["matched_metadata"].mean())
print("Activity coverage:", df["matched_activity"].mean())
print("High-demand distribution:")
print(df["high_demand"].value_counts(dropna=False))
display(df.head())


In [ ]:
# ============================================================
# 5. Feature sets and modeling utilities
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score, balanced_accuracy_score,
    accuracy_score, confusion_matrix, brier_score_loss, precision_recall_curve,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.calibration import calibration_curve

try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

def make_preprocessor(X: pd.DataFrame):
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols)
    ])

def get_tree_classifier():
    if HAS_XGB:
        return XGBClassifier(
            n_estimators=250,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    return RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )

def get_tree_regressor():
    if HAS_XGB:
        return XGBRegressor(
            n_estimators=250,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    return RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

accommodation_features = [
    "month", "provider_prefix", "country_clean", "city_clean",
    "final_price_aud", "guest_rating", "guest_rating_count",
    "availability_score", "popularity", "star_rating_score",
    "log_price", "rating_x_price", "availability_x_popularity"
]
integrated_features = accommodation_features + [
    "activity_count", "avg_activity_price", "avg_activity_rating",
    "avg_activity_availability", "total_activity_views",
    "dominant_activity_category"
]

accommodation_features = [c for c in accommodation_features if c in df.columns]
integrated_features = [c for c in integrated_features if c in df.columns]

print("Accommodation-only features:", accommodation_features)
print("Integrated features:", integrated_features)


## 2. Classification calibration and threshold-sensitivity analysis

Cell berikut menghasilkan tabel dan gambar untuk mengganti placeholder:

- `calibration_threshold_results.csv`
- `fig_calibration_threshold_diagnostics.png`
- `fig_precision_recall_threshold_curve.png`


In [ ]:
# ============================================================
# 6. Classification benchmark + calibration and threshold sensitivity
# ============================================================

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (y_prob >= lo) & (y_prob < hi if i < n_bins - 1 else y_prob <= hi)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

def build_classifier_pipeline(model, X):
    return Pipeline([
        ("prep", make_preprocessor(X)),
        ("model", model)
    ])

def evaluate_classifier(name, feature_set_name, X_train, X_test, y_train, y_test, model):
    pipe = build_classifier_pipeline(model, X_train)
    pipe.fit(X_train, y_train)
    if hasattr(pipe.named_steps["model"], "predict_proba"):
        prob = pipe.predict_proba(X_test)[:, 1]
    else:
        pred = pipe.predict(X_test)
        prob = pred.astype(float)
    pred = (prob >= 0.5).astype(int)
    return pipe, {
        "feature_set": feature_set_name,
        "model": name,
        "roc_auc": roc_auc_score(y_test, prob) if len(np.unique(y_test)) > 1 else np.nan,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "brier_score": brier_score_loss(y_test, prob),
        "ece": expected_calibration_error(y_test, prob),
    }, prob

# Prepare classification split
X_acc = df[accommodation_features].copy()
X_int = df[integrated_features].copy()
y_cls = df["high_demand"].astype(int)

X_acc_train, X_acc_test, y_train, y_test = train_test_split(
    X_acc, y_cls, test_size=0.30, random_state=RANDOM_STATE, stratify=y_cls
)
X_int_train, X_int_test, _, _ = train_test_split(
    X_int, y_cls, test_size=0.30, random_state=RANDOM_STATE, stratify=y_cls
)

models_cls = {
    "Dummy": DummyClassifier(strategy="prior"),
    "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Tree": get_tree_classifier()
}

classification_results = []
classification_artifacts = {}

for feature_set_name, Xtr, Xte in [
    ("accommodation_only", X_acc_train, X_acc_test),
    ("integrated", X_int_train, X_int_test),
]:
    for model_name, model in models_cls.items():
        fitted, metrics, prob = evaluate_classifier(model_name, feature_set_name, Xtr, Xte, y_train, y_test, model)
        classification_results.append(metrics)
        classification_artifacts[(feature_set_name, model_name)] = {
            "pipeline": fitted, "prob": prob, "X_test": Xte, "y_test": y_test
        }

classification_results_df = pd.DataFrame(classification_results)
classification_results_df.to_csv(TABLE_DIR / "classification_benchmark_results.csv", index=False)
display(classification_results_df.sort_values("roc_auc", ascending=False))

# Calibration and threshold table for selected models
selected_keys = [
    ("accommodation_only", "Tree"),
    ("integrated", "Tree"),
    ("integrated", "LogReg"),
]

threshold_rows = []
threshold_grid = np.round(np.arange(0.05, 0.96, 0.05), 2)

for key in selected_keys:
    if key not in classification_artifacts:
        continue
    prob = classification_artifacts[key]["prob"]
    y_ref = classification_artifacts[key]["y_test"]
    for th in threshold_grid:
        pred = (prob >= th).astype(int)
        threshold_rows.append({
            "feature_set": key[0],
            "model": key[1],
            "threshold": th,
            "precision": precision_score(y_ref, pred, zero_division=0),
            "recall": recall_score(y_ref, pred, zero_division=0),
            "f1": f1_score(y_ref, pred, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_ref, pred),
            "positive_rate_pred": pred.mean(),
            "brier_score": brier_score_loss(y_ref, prob),
            "ece": expected_calibration_error(y_ref, prob)
        })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(TABLE_DIR / "classification_threshold_sensitivity.csv", index=False)

# Summary table: best F1 threshold by selected model
best_threshold_df = (
    threshold_df.sort_values(["feature_set", "model", "f1"], ascending=[True, True, False])
    .groupby(["feature_set", "model"])
    .head(1)
    .reset_index(drop=True)
)
best_threshold_df.to_csv(TABLE_DIR / "calibration_threshold_results.csv", index=False)
display(best_threshold_df)

# Figure 1: reliability diagram
plt.figure(figsize=(6, 4))
for key in selected_keys:
    if key not in classification_artifacts:
        continue
    prob = classification_artifacts[key]["prob"]
    y_ref = classification_artifacts[key]["y_test"]
    frac_pos, mean_pred = calibration_curve(y_ref, prob, n_bins=10, strategy="uniform")
    plt.plot(mean_pred, frac_pos, marker="o", label=f"{key[0]}-{key[1]}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed positive rate")
plt.title("Reliability Diagram")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_reliability_diagram.png", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR / "fig_calibration_threshold_diagnostics.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 2: threshold sensitivity
plt.figure(figsize=(6, 4))
for key in selected_keys:
    tmp = threshold_df[(threshold_df["feature_set"] == key[0]) & (threshold_df["model"] == key[1])]
    if tmp.empty:
        continue
    plt.plot(tmp["threshold"], tmp["precision"], label=f"{key[0]}-{key[1]} precision")
    plt.plot(tmp["threshold"], tmp["recall"], linestyle="--", label=f"{key[0]}-{key[1]} recall")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title("Precision--Recall by Threshold")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_precision_recall_threshold_curve.png", dpi=300, bbox_inches="tight")
plt.show()


## 3. Temporal holdout, robustness, and drift diagnostics

Cell berikut menguji apakah hasil tetap stabil saat train/test dipisahkan berdasarkan waktu. Jika hanya ada sedikit bulan, notebook akan tetap menghasilkan file output dengan catatan `status`.


In [ ]:
# ============================================================
# 7. Temporal holdout and feature drift diagnostics
# ============================================================

def regression_metrics(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred, squared=False),
        "r2": r2_score(y_true, y_pred)
    }

def classification_metrics(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan,
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred)
    }

def build_regressor_pipeline(model, X):
    return Pipeline([
        ("prep", make_preprocessor(X)),
        ("model", model)
    ])

def psi_numeric(expected, actual, bins=10):
    expected = pd.to_numeric(pd.Series(expected), errors="coerce").dropna()
    actual = pd.to_numeric(pd.Series(actual), errors="coerce").dropna()
    if len(expected) < 5 or len(actual) < 5:
        return np.nan
    try:
        quantiles = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
        if len(quantiles) <= 2:
            return np.nan
        e_counts, _ = np.histogram(expected, bins=quantiles)
        a_counts, _ = np.histogram(actual, bins=quantiles)
        e_pct = np.maximum(e_counts / max(e_counts.sum(), 1), 1e-6)
        a_pct = np.maximum(a_counts / max(a_counts.sum(), 1), 1e-6)
        return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))
    except Exception:
        return np.nan

temporal_rows = []
drift_rows = []

months = sorted(df["payment_month"].dropna().unique())
if len(months) < 2:
    temporal_rows.append({
        "task": "temporal_holdout",
        "feature_set": "integrated",
        "model": "Tree",
        "status": "skipped_less_than_two_months",
        "metric": np.nan,
        "value": np.nan
    })
else:
    cutoff = months[-1]
    train_mask = df["payment_month"] < cutoff
    test_mask = df["payment_month"] == cutoff

    temporal_train = df[train_mask].copy()
    temporal_test = df[test_mask].copy()

    print("Temporal train:", temporal_train.shape, "Temporal test:", temporal_test.shape, "Cutoff month:", cutoff)

    X_train_t = temporal_train[integrated_features]
    X_test_t = temporal_test[integrated_features]
    y_cls_train_t = temporal_train["high_demand"].astype(int)
    y_cls_test_t = temporal_test["high_demand"].astype(int)
    y_reg_train_t = temporal_train["number_transactions"].astype(float)
    y_reg_test_t = temporal_test["number_transactions"].astype(float)

    # Classification temporal holdout
    if len(np.unique(y_cls_train_t)) > 1 and len(np.unique(y_cls_test_t)) > 1:
        cls_pipe = build_classifier_pipeline(get_tree_classifier(), X_train_t)
        cls_pipe.fit(X_train_t, y_cls_train_t)
        prob_t = cls_pipe.predict_proba(X_test_t)[:, 1]
        m = classification_metrics(y_cls_test_t, prob_t)
        for k, v in m.items():
            temporal_rows.append({
                "task": "classification",
                "feature_set": "integrated",
                "model": "Tree",
                "split": "temporal_holdout",
                "metric": k,
                "value": v,
                "status": "ok"
            })
    else:
        temporal_rows.append({
            "task": "classification",
            "feature_set": "integrated",
            "model": "Tree",
            "split": "temporal_holdout",
            "metric": np.nan,
            "value": np.nan,
            "status": "skipped_single_class_in_train_or_test"
        })

    # Regression temporal holdout
    reg_pipe = build_regressor_pipeline(get_tree_regressor(), X_train_t)
    reg_pipe.fit(X_train_t, y_reg_train_t)
    pred_reg_t = reg_pipe.predict(X_test_t)
    m = regression_metrics(y_reg_test_t, pred_reg_t)
    for k, v in m.items():
        temporal_rows.append({
            "task": "regression",
            "feature_set": "integrated",
            "model": "Tree",
            "split": "temporal_holdout",
            "metric": k,
            "value": v,
            "status": "ok"
        })

    # Numeric drift based on PSI
    for col in integrated_features:
        if col in X_train_t.columns and pd.api.types.is_numeric_dtype(X_train_t[col]):
            drift_rows.append({
                "feature": col,
                "psi": psi_numeric(X_train_t[col], X_test_t[col]),
                "train_mean": pd.to_numeric(X_train_t[col], errors="coerce").mean(),
                "test_mean": pd.to_numeric(X_test_t[col], errors="coerce").mean()
            })

temporal_df = pd.DataFrame(temporal_rows)
drift_df = pd.DataFrame(drift_rows).sort_values("psi", ascending=False) if drift_rows else pd.DataFrame(columns=["feature", "psi", "train_mean", "test_mean"])

temporal_df.to_csv(TABLE_DIR / "temporal_holdout_robustness_results.csv", index=False)
drift_df.to_csv(TABLE_DIR / "feature_drift_psi_results.csv", index=False)

display(temporal_df)
display(drift_df.head(20))

# Figure: temporal metrics
if not temporal_df.empty and "value" in temporal_df.columns and temporal_df["value"].notna().any():
    plot_df = temporal_df.dropna(subset=["value"]).copy()
    plot_df["label"] = plot_df["task"] + " - " + plot_df["metric"]
    plt.figure(figsize=(6, 4))
    plt.barh(plot_df["label"], plot_df["value"])
    plt.xlabel("Metric value")
    plt.title("Temporal Holdout Performance")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_temporal_holdout_performance.png", dpi=300, bbox_inches="tight")
    plt.show()

# Figure: PSI drift
if not drift_df.empty and drift_df["psi"].notna().any():
    top_drift = drift_df.dropna(subset=["psi"]).head(15)
    plt.figure(figsize=(6, 4))
    plt.barh(top_drift["feature"][::-1], top_drift["psi"][::-1])
    plt.xlabel("Population Stability Index (PSI)")
    plt.title("Top Feature Drift Diagnostics")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_feature_drift_psi.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIG_DIR / "fig_temporal_drift_diagnostics.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    # Create placeholder image so the paper path can still be tested.
    plt.figure(figsize=(6, 2))
    plt.text(0.5, 0.5, "Temporal drift diagnostics skipped\\n(insufficient temporal split)", ha="center", va="center")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_temporal_drift_diagnostics.png", dpi=300, bbox_inches="tight")
    plt.show()


## 4. Recommendation baseline comparison

Cell berikut membandingkan proposed context-aware score dengan baseline:

1. **Popularity-only**
2. **Rating-only**
3. **Geo-popularity**
4. **Proposed**


In [ ]:
# ============================================================
# 8. Recommendation baseline comparison
# ============================================================

# Prepare hotel sample: one row per property with destination and price.
hotel_cols = ["property_id_clean", "country_clean", "city_clean", "final_price_aud"]
hotel_rows = df.dropna(subset=["country_clean", "city_clean"]).drop_duplicates("property_id_clean")[hotel_cols].copy()
hotel_rows["hotel_price"] = pd.to_numeric(hotel_rows["final_price_aud"], errors="coerce")

# Limit for speed if dataset is large
MAX_HOTELS_FOR_RECOMMENDATION_EVAL = 1000
if len(hotel_rows) > MAX_HOTELS_FOR_RECOMMENDATION_EVAL:
    hotel_rows = hotel_rows.sample(MAX_HOTELS_FOR_RECOMMENDATION_EVAL, random_state=RANDOM_STATE)

# Prepare activity scoring table
activities = act.copy()
activities["rating_norm"] = minmax(activities["activity_rating"]).fillna(0)
activities["views_norm"] = minmax(activities["activity_views"]).fillna(0)
activities["availability_norm"] = minmax(activities["activity_availability"]).fillna(0)
activities["price_norm"] = minmax(activities["activity_price"]).fillna(0)
activities["activity_category"] = activities["activity_category"].fillna("unknown").astype(str)

global_candidates = activities.copy()
if len(global_candidates) > 50000:
    # Keep high-signal candidates to reduce computation.
    global_candidates = global_candidates.sort_values(["views_norm", "rating_norm"], ascending=False).head(50000)

def topk_for_hotel(hotel, strategy="proposed", k=5):
    country = str(hotel["country_clean"]).lower()
    city = str(hotel["city_clean"]).lower()
    hotel_price = hotel.get("hotel_price", np.nan)

    cand = global_candidates.copy()
    cand["same_city"] = ((cand["country_clean"] == country) & (cand["city_clean"] == city)).astype(float)
    cand["same_country"] = (cand["country_clean"] == country).astype(float)

    # Prefer same city/country candidates if available.
    if cand["same_city"].sum() >= k:
        cand = cand[cand["same_city"] == 1].copy()
    elif cand["same_country"].sum() >= k:
        cand = cand[cand["same_country"] == 1].copy()

    if pd.notna(hotel_price):
        price_gap = (pd.to_numeric(cand["activity_price"], errors="coerce") - hotel_price).abs()
        cand["price_fit"] = 1 - minmax(price_gap).fillna(0)
    else:
        cand["price_fit"] = 0.5

    if strategy == "popularity_only":
        cand["score"] = cand["views_norm"]
    elif strategy == "rating_only":
        cand["score"] = cand["rating_norm"]
    elif strategy == "geo_popularity":
        cand["score"] = 0.60 * cand["same_city"] + 0.40 * cand["views_norm"]
    elif strategy == "proposed":
        cand["score"] = (
            0.35 * cand["same_city"]
            + 0.25 * cand["rating_norm"]
            + 0.20 * cand["views_norm"]
            + 0.15 * cand["availability_norm"]
            + 0.05 * cand["price_fit"]
        )
    else:
        raise ValueError(strategy)

    return cand.sort_values("score", ascending=False).head(k).copy()

strategies = ["popularity_only", "rating_only", "geo_popularity", "proposed"]
rec_rows = []

for strategy in strategies:
    for _, hotel in hotel_rows.iterrows():
        topk = topk_for_hotel(hotel, strategy=strategy, k=5)
        if topk.empty:
            continue
        rec_rows.append({
            "strategy": strategy,
            "property_id": hotel["property_id_clean"],
            "same_city_rate": topk["same_city"].mean(),
            "mean_rating": pd.to_numeric(topk["activity_rating"], errors="coerce").mean(),
            "mean_views": pd.to_numeric(topk["activity_views"], errors="coerce").mean(),
            "mean_availability": pd.to_numeric(topk["activity_availability"], errors="coerce").mean(),
            "category_diversity": topk["activity_category"].nunique(),
            "mean_score": topk["score"].mean()
        })

rec_detail_df = pd.DataFrame(rec_rows)
rec_detail_df.to_csv(TABLE_DIR / "recommendation_baseline_detail.csv", index=False)

if rec_detail_df.empty:
    rec_summary_df = pd.DataFrame(columns=["strategy", "same_city_rate", "mean_rating", "mean_views", "mean_availability", "category_diversity", "mean_score", "proxy_score"])
else:
    rec_summary_df = rec_detail_df.groupby("strategy").agg(
        same_city_rate=("same_city_rate", "mean"),
        mean_rating=("mean_rating", "mean"),
        mean_views=("mean_views", "mean"),
        mean_availability=("mean_availability", "mean"),
        category_diversity=("category_diversity", "mean"),
        mean_score=("mean_score", "mean")
    ).reset_index()

    # Normalize metrics for a composite proxy score.
    for col in ["same_city_rate", "mean_rating", "mean_views", "mean_availability", "category_diversity"]:
        rec_summary_df[col + "_norm"] = minmax(rec_summary_df[col]).fillna(0)
    rec_summary_df["proxy_score"] = (
        0.30 * rec_summary_df["same_city_rate_norm"]
        + 0.20 * rec_summary_df["mean_rating_norm"]
        + 0.20 * rec_summary_df["mean_views_norm"]
        + 0.15 * rec_summary_df["mean_availability_norm"]
        + 0.15 * rec_summary_df["category_diversity_norm"]
    )

rec_summary_df.to_csv(TABLE_DIR / "recommendation_baseline_comparison.csv", index=False)
display(rec_summary_df)

# Figure: recommendation baseline comparison
if not rec_summary_df.empty:
    metrics_to_plot = ["same_city_rate", "mean_rating_norm", "mean_views_norm", "category_diversity_norm", "proxy_score"]
    plot_df = rec_summary_df.set_index("strategy")[metrics_to_plot]
    ax = plot_df.plot(kind="bar", figsize=(7, 4))
    plt.ylabel("Normalized / rate value")
    plt.title("Recommendation Baseline Comparison")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_recommendation_baseline_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    plt.figure(figsize=(6, 2))
    plt.text(0.5, 0.5, "Recommendation baseline comparison skipped\\n(no valid candidates)", ha="center", va="center")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_recommendation_baseline_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()


## 5. Export paper-ready manifest

Cell terakhir membuat manifest output sehingga Anda dapat mengetahui tabel dan gambar mana yang harus dimasukkan ke paper.


In [ ]:
# ============================================================
# 9. Export manifest
# ============================================================

manifest = {
    "tables": sorted([str(p.relative_to(OUTPUT_DIR)) for p in TABLE_DIR.glob("*.csv")]),
    "figures": sorted([str(p.relative_to(OUTPUT_DIR)) for p in FIG_DIR.glob("*.png")]),
    "paper_placeholder_mapping": {
        "Table calibration_placeholder": "tables/calibration_threshold_results.csv",
        "Figure calibration_placeholder": "figures/fig_calibration_threshold_diagnostics.png and figures/fig_precision_recall_threshold_curve.png",
        "Table temporal_robustness_placeholder": "tables/temporal_holdout_robustness_results.csv",
        "Figure temporal_drift_placeholder": "figures/fig_temporal_drift_diagnostics.png",
        "Table recommendation_baseline_placeholder": "tables/recommendation_baseline_comparison.csv",
        "Additional recommendation figure": "figures/fig_recommendation_baseline_comparison.png"
    }
}

with open(OUTPUT_DIR / "revision_experiment_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print("\nDone. Outputs saved to:", OUTPUT_DIR.resolve())
